### Create Eval config

In [1]:
import sys

sys.path.append("/pscratch/sd/s/suryad/Ocean_Emulator/src")

In [2]:
# TODO:
# - resubmit jobs / preempted job safety
# - better stepper module and a cleaner model module
# - cleaner dataset modules
import argparse
import datetime
import gc
import logging
import os
import time
import traceback
import warnings
from functools import partial
from pathlib import Path
from typing import Union

import dask
import numpy as np
import torch
import torch.nn as nn
import xarray as xr
from torch.utils.data import (
    ConcatDataset,
    DataLoader,
    Dataset,
    DistributedSampler,
    RandomSampler,
)

from aggregator import Aggregator, LossAggregator
from config import EvalConfig
from constants import EXTRA_VARS, INPT_VARS, OUT_VARS, TensorMap, construct_metadata
from datasets import InferenceDataset, TrainDataset
from models.unet import UNet
from stepper import Stepper, TrainOutput, ValOutput
from utils.data import Normalize, extract_wet_mask, get_time_slice
from utils.device import get_device, using_gpu
from utils.distributed import (
    all_reduce_mean,
    init_distributed_mode,
    is_main_process,
    set_seed,
)
from utils.logging import MetricLogger, SmoothedValue, handle_logging, handle_warnings
from utils.loss import (
    decomposed_mse,
    decomposed_mse_cos_weighted,
    decomposed_mse_diff_weighted,
    decomposed_mse_mae,
    decomposed_mse_scaled,
)
from utils.model import get_model_summary
from utils.train import collate_train_data
from utils.wandb import WandBLogger


class Eval:
    def __init__(self, cfg) -> None:
        if not using_gpu():
            logging.info("No GPU available, using CPU")
            cfg.distributed.enabled = False

        self.device = get_device()

        # Adjust workers and memory pinning based on device
        if not using_gpu():
            cfg.data.num_workers = 0  # Disable multi-processing on CPU
            cfg.pin_mem = False
        elif cfg.disk_mode:
            cfg.data.num_workers = torch.cuda.device_count() * cfg.data.num_workers
            cfg.pin_mem = True

        # Set seeds
        set_seed(cfg.experiment.rand_seed)

        # Getting input, extra input and output
        self.inputs = INPT_VARS[cfg.experiment.exp_num_in]
        self.extra_in = EXTRA_VARS[cfg.experiment.exp_num_extra]
        self.outputs = OUT_VARS[cfg.experiment.exp_num_out]

        levels = cfg.experiment.exp_num_in.split("_")[-1]
        if "all" in levels:
            self.levels = 19
        elif "2D" in levels:
            self.levels = 1
        else:
            self.levels = int(levels)

        self.str_in = "".join([i + "_" for i in self.inputs])
        self.str_ext = "".join([i + "_" for i in self.extra_in])
        self.str_out = "".join([i + "_" for i in self.outputs])

        logging.info(f"inputs: {self.str_in}")
        logging.info(f"extra inputs: {self.str_ext}")
        logging.info(f"outputs: {self.str_out}")
        logging.info(f"levels: {self.levels}")

        self.N_atm = len(self.extra_in)
        self.N_in = len(self.inputs)
        self.N_extra = self.N_atm  # Number of atmosphere variables
        self.N_out = len(self.outputs)

        self.num_in = int((cfg.data.hist + 1) * self.N_in + self.N_extra)
        self.num_out = int((cfg.data.hist + 1) * len(self.outputs))

        self.tensor_map = TensorMap.init_instance(cfg.experiment.exp_num_out)

        logging.info(f"Number of inputs: {self.num_in}")
        logging.info(f"Number of outputs: {self.num_out}")

        # Dataloaders
        logging.info(f"Loading data")
        assert cfg.data.depth_mode == "surface" or cfg.data.depth_mode == "all"
        self.data_dir = cfg.experiment.data_dir
        self.wet_file = cfg.data.wet_file
        self.data_path = cfg.data.data_path
        self.data_means_path = cfg.data.data_means_path
        self.data_stds_path = cfg.data.data_stds_path
        self.scaling_residuals_file = cfg.data.scaling_residuals_file

        if "*" in self.data_path:
            self.data = xr.open_mfdataset(
                os.path.join(self.data_dir, self.data_path),
                engine="netcdf4",
                chunks={"time": 1, "lat": 180, "lon": 360},
            )
        else:
            self.data = xr.open_zarr(os.path.join(self.data_dir, self.data_path))
        self.data_mean = xr.open_dataset(
            os.path.join(self.data_dir, self.data_means_path), engine="netcdf4"
        )
        self.data_std = xr.open_dataset(
            os.path.join(self.data_dir, self.data_stds_path), engine="netcdf4"
        )

        self.metadata = construct_metadata(self.data)
        wet_zarr = xr.open_zarr(os.path.join(self.data_dir, self.wet_file))
        self.wet = extract_wet_mask(wet_zarr, self.outputs, cfg.data.hist)
        wet_without_hist = extract_wet_mask(wet_zarr, self.outputs, 0)
        areacello = wet_zarr.areacello.to_numpy()
        self.area_weights = areacello / areacello.sum()
        self.area_weights = torch.from_numpy(self.area_weights).to(self.device)

        self.normalize = Normalize.init_instance(
            self.data_mean,
            self.data_std,
            self.inputs,
            self.extra_in,
            self.outputs,
            wet_without_hist,
        )

        # Model
        logging.info(f"Getting model {cfg.experiment.network}")
        if "convnextunet" == cfg.experiment.network:
            if cfg.unet.ch_width[0] != self.num_in:
                logging.info(
                    f"NOTE: Changing input channels to match data "
                    f"{cfg.unet.ch_width[0]}->{self.num_in}"
                )
                cfg.unet.ch_width[0] = self.num_in
            if cfg.unet.n_out != self.num_out:
                logging.info(
                    f"NOTE: Changing output channels to match data "
                    f"{cfg.unet.n_out}->{self.num_out}"
                )
                cfg.unet.n_out = self.num_out
            model = UNet(cfg.unet, hist=cfg.data.hist, wet=self.wet.to(self.device)).to(
                self.device
            )
        else:
            raise NotImplementedError

        get_model_summary(model, self.num_in)

        self.model = model
        self.nets_dir = cfg.experiment.nets_dir
        self.network = cfg.experiment.network

        # Initialize WandB
        self.wandb_logger = WandBLogger.init_instance()
        self.wandb_logger.configure(
            cfg.experiment.wandb.mode == "online", is_main_process()
        )

        # Set up wandb run
        self.wandb_id, self.wandb_name = self.wandb_logger.setup_run(
            None, cfg, finetune=False
        )

        # Training
        self.hist = cfg.data.hist
        self.output_dir = cfg.experiment.output_dir
        self.network = cfg.experiment.network
        self.debug = cfg.debug
        self.num_workers = cfg.data.num_workers
        self.inference_time = cfg.inference
        self.time_delta = cfg.data.time_delta

        self.init_inference_store()

    def init_inference_store(self):
        time_slice_with_initial_condition, self.num_time_steps = get_time_slice(
            self.inference_time,
            time_delta=self.time_delta,
            hist=self.hist,
        )
        inference_data = self.data.sel(time=time_slice_with_initial_condition)
        self.inference_dataset = InferenceDataset(
            inference_data,
            self.inputs,
            self.extra_in,
            self.outputs,
            self.wet,
            self.hist,
            long_rollout=True,
        )

    def run(self) -> None:
        start_time = time.time()
        inf_stats = self.inference_one_epoch(0)
        time_elapsed = time.time() - start_time

        log_stats = {
            **inf_stats,
            "eval_total_seconds": time_elapsed,
        }

        if is_main_process():
            self.wandb_logger.log(log_stats, step=0)

        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        logging.info(f"Eval time (Including wandb logging) {total_time_str}")
        self.finish()

    @torch.no_grad()
    def inference_one_epoch(self, epoch):
        self.model.eval()
        inf_aggregator = Aggregator.get_inference_aggregator(
            self.num_time_steps,
            self.metadata,
            self.hist,
            self.area_weights,
            self.num_out,
        )

        Stepper.inline_inference(
            self.model,
            self.inference_dataset,
            inf_aggregator,
            epoch,
        )
        logs = inf_aggregator.get_summary_logs()
        return {f"inference/{k}": v for k, v in logs.items()}

    def load_checkpoint(self, checkpoint_path, finetune=False):
        logging.info(f"Loaded checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        if finetune:
            self.model.load_state_dict(checkpoint["model"])
        else:
            self.optimizer.load_state_dict(checkpoint["optimizer"])
            if self.scheduler:
                self.scheduler.load_state_dict(checkpoint["scheduler"])
            self.start_epoch = checkpoint["epoch"] + 1
            self.wandb_id = checkpoint["wandb_id"]
            self.wandb_name = checkpoint["wandb_name"]

            logging.info(f"Start Epoch: {self.start_epoch}")
            logging.info(f"Wandb id: {self.wandb_id}")
            logging.info(f"Wandb name: {self.wandb_name}")
            logging.info(f"Optimizer LR: {self.optimizer.param_groups[-1]['lr']}")

    def finish(self):
        self.wandb_logger.finish()

In [3]:
subname = "test-eval"
config = "../configs/eval_cm4_thermo.yaml"

overrides = {}
if subname != "":
    overrides["sub_name"] = subname

# Load config from YAML
cfg = EvalConfig.from_yaml(config, overrides)

if not os.path.exists(cfg.experiment.output_dir):
    os.makedirs(cfg.experiment.output_dir, exist_ok=True)


Evaluator = Eval(cfg)

In [4]:
Evaluator.run()

OutOfMemoryError: CUDA out of memory. Tried to allocate 198.00 MiB. GPU 0 has a total capacity of 39.38 GiB of which 92.69 MiB is free. Process 404258 has 0 bytes memory in use. Process 548762 has 11.29 GiB memory in use. Process 774217 has 13.61 GiB memory in use. Including non-PyTorch memory, this process has 8.17 GiB memory in use. Of the allocated memory 6.76 GiB is allocated by PyTorch, and 945.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Copy code from train_3D.py and make changes to inference only